# Export CycleGAN Pretrained Generators to ONNX

Converts the four official CycleGAN photo→painting generators to ONNX format
so they can be used in the PetersPictureStyler app.

**Run on Kaggle** (free T4 GPU, PyTorch pre-installed).

## What this notebook does
1. Downloads the four pretrained `.pth` state-dicts from the official server
2. Instantiates the ResNet-9 generator architecture (self-contained below)
3. Exports each generator to ONNX with dynamic H/W axes
4. Verifies inference on a random test tensor
5. Lists the output files for download

## Models produced
| ONNX file | Style | Size |
|-----------|-------|------|
| `style_monet.onnx`   | Monet impressionist paintings | ~11 MB |
| `style_vangogh.onnx` | Van Gogh post-impressionism   | ~11 MB |
| `style_cezanne.onnx` | Cézanne post-impressionism    | ~11 MB |
| `style_ukiyoe.onnx`  | Japanese Ukiyo-e woodblock    | ~11 MB |

After downloading, install each via `add_GAN_style.ipynb` with layout **`nchw_tanh`**.

---
Source models: https://github.com/junyanz/pytorch-CycleGAN-and-pix2pix  
License: BSD (see https://github.com/junyanz/pytorch-CycleGAN-and-pix2pix/blob/master/LICENSE)

In [ ]:
import functools
import urllib.request
from pathlib import Path

import torch
import torch.nn as nn
import numpy as np
from PIL import Image

print(f"PyTorch {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

## ResNet-9 Generator

Self-contained copy of `ResnetGenerator` + `ResnetBlock` from the official repo
(`junyanz/pytorch-CycleGAN-and-pix2pix/models/networks.py`).
No dependency on the full CycleGAN codebase.

In [ ]:
class ResnetBlock(nn.Module):
    """A single ResNet residual block used inside ResnetGenerator."""

    def __init__(self, dim: int, padding_type: str, norm_layer, use_dropout: bool, use_bias: bool):
        super().__init__()
        self.conv_block = self._build(dim, padding_type, norm_layer, use_dropout, use_bias)

    def _build(self, dim, padding_type, norm_layer, use_dropout, use_bias):
        block = []
        for _ in range(2):  # two conv layers per block
            p = 0
            if padding_type == "reflect":
                block += [nn.ReflectionPad2d(1)]
            elif padding_type == "replicate":
                block += [nn.ReplicationPad2d(1)]
            elif padding_type == "zero":
                p = 1
            else:
                raise NotImplementedError(f"padding '{padding_type}' not supported")
            block += [nn.Conv2d(dim, dim, kernel_size=3, padding=p, bias=use_bias),
                      norm_layer(dim)]
            if _ == 0:  # ReLU + optional dropout only after first conv
                block += [nn.ReLU(True)]
                if use_dropout:
                    block += [nn.Dropout(0.5)]
        return nn.Sequential(*block)

    def forward(self, x):
        return x + self.conv_block(x)  # residual connection


class ResnetGenerator(nn.Module):
    """ResNet-based generator (6 or 9 blocks).

    Architecture: Reflect-pad → Conv7 → [down×2] → [ResBlocks] → [up×2] → Conv7 → Tanh
    Matches the official CycleGAN checkpoint format exactly.
    """

    def __init__(
        self,
        input_nc: int = 3,
        output_nc: int = 3,
        ngf: int = 64,
        norm_layer=None,
        use_dropout: bool = False,
        n_blocks: int = 9,
        padding_type: str = "reflect",
    ):
        assert n_blocks >= 0
        super().__init__()
        if norm_layer is None:
            norm_layer = functools.partial(nn.InstanceNorm2d, affine=False, track_running_stats=False)

        use_bias = (
            norm_layer.func == nn.InstanceNorm2d
            if isinstance(norm_layer, functools.partial)
            else norm_layer == nn.InstanceNorm2d
        )

        model = [
            nn.ReflectionPad2d(3),
            nn.Conv2d(input_nc, ngf, kernel_size=7, padding=0, bias=use_bias),
            norm_layer(ngf),
            nn.ReLU(True),
        ]

        # Downsampling
        n_down = 2
        for i in range(n_down):
            mult = 2 ** i
            model += [
                nn.Conv2d(ngf * mult, ngf * mult * 2, kernel_size=3, stride=2, padding=1, bias=use_bias),
                norm_layer(ngf * mult * 2),
                nn.ReLU(True),
            ]

        # ResNet blocks
        mult = 2 ** n_down
        for _ in range(n_blocks):
            model += [ResnetBlock(ngf * mult, padding_type=padding_type,
                                  norm_layer=norm_layer, use_dropout=use_dropout, use_bias=use_bias)]

        # Upsampling
        for i in range(n_down):
            mult = 2 ** (n_down - i)
            model += [
                nn.ConvTranspose2d(ngf * mult, ngf * mult // 2,
                                   kernel_size=3, stride=2, padding=1, output_padding=1, bias=use_bias),
                norm_layer(ngf * mult // 2),
                nn.ReLU(True),
            ]

        model += [
            nn.ReflectionPad2d(3),
            nn.Conv2d(ngf, output_nc, kernel_size=7, padding=0),
            nn.Tanh(),
        ]

        self.model = nn.Sequential(*model)

    def forward(self, x):
        return self.model(x)


# Quick sanity check: forward pass on random data
_test_net = ResnetGenerator(n_blocks=9).eval()
with torch.no_grad():
    _out = _test_net(torch.zeros(1, 3, 256, 256))
assert _out.shape == (1, 3, 256, 256), f"Unexpected shape: {_out.shape}"
assert _out.min() >= -1.0 and _out.max() <= 1.0, "Output not in [-1, 1]"
del _test_net
print("ResnetGenerator architecture OK — output shape (1,3,256,256), range [-1, 1]")

## Download the four pretrained `.pth` state-dicts

Primary URL: `http://efrosgans.eecs.berkeley.edu/cyclegan/pretrained_models/`  
Fallback: HuggingFace mirror (if primary is unreachable)

In [ ]:
MODELS = ["style_monet", "style_vangogh", "style_cezanne", "style_ukiyoe"]

PRIMARY_URL   = "http://efrosgans.eecs.berkeley.edu/cyclegan/pretrained_models/{name}.pth"
FALLBACK_URL  = "https://huggingface.co/tmabraham/{name}_cyclegan/resolve/main/latest_net_G.pth"

PTH_DIR = Path("/kaggle/working/pth")
PTH_DIR.mkdir(parents=True, exist_ok=True)


def _download(url: str, dest: Path) -> bool:
    """Attempt to download url → dest. Returns True on success."""
    try:
        print(f"  Trying {url} ...", end=" ", flush=True)
        urllib.request.urlretrieve(url, dest)
        size_mb = dest.stat().st_size / 1_048_576
        print(f"OK ({size_mb:.1f} MB)")
        return True
    except Exception as exc:
        print(f"FAILED — {exc}")
        return False


pth_paths: dict[str, Path] = {}

for name in MODELS:
    dest = PTH_DIR / f"{name}.pth"
    print(f"Downloading {name} ...")
    if _download(PRIMARY_URL.format(name=name), dest):
        pth_paths[name] = dest
    elif _download(FALLBACK_URL.format(name=name), dest):
        pth_paths[name] = dest
    else:
        print(f"  *** Could not download {name} — skipping ***")

print(f"\nDownloaded {len(pth_paths)}/{len(MODELS)} models.")

## Export each `.pth` to ONNX

- Input/output: `[1, 3, H, W]` float32, range `[-1, 1]` (`nchw_tanh` layout)
- Dynamic axes on H and W so the model accepts any tile size at runtime
- Opset 17 (supported by `onnxruntime` 1.16+)

In [ ]:
ONNX_DIR = Path("/kaggle/working/onnx")
ONNX_DIR.mkdir(parents=True, exist_ok=True)

# InstanceNorm2d without running-stats tracking (inference only).
# The pretrained checkpoints were saved with track_running_stats=True,
# so those buffers appear as "unexpected keys" — we load with strict=False
# to silently discard them.  They are unused at inference time.
_NORM = functools.partial(nn.InstanceNorm2d, affine=False, track_running_stats=False)


def export_to_onnx(name: str, pth_path: Path) -> Path:
    """Load a CycleGAN .pth state-dict and export to ONNX."""
    print(f"\nExporting '{name}' ...")

    # --- Build the generator skeleton ---
    net = ResnetGenerator(
        input_nc=3, output_nc=3, ngf=64,
        norm_layer=_NORM, use_dropout=False, n_blocks=9,
    )

    # --- Load weights ---
    # weights_only=False needed for older-format state dicts; safe here because
    # files come from the official Berkeley server / HuggingFace.
    # strict=False: the checkpoint contains running_mean/running_var/num_batches_tracked
    # buffers from InstanceNorm2d(track_running_stats=True).  Our inference model
    # uses track_running_stats=False so those buffers don't exist — they are
    # safely discarded.  No learnable parameters are missing.
    state = torch.load(pth_path, map_location="cpu", weights_only=False)
    incompatible = net.load_state_dict(state, strict=False)
    if incompatible.missing_keys:
        print(f"  WARNING — missing keys (check these!): {incompatible.missing_keys}")
    unexpected = [k for k in incompatible.unexpected_keys
                  if not k.endswith(("running_mean", "running_var", "num_batches_tracked"))]
    if unexpected:
        print(f"  WARNING — truly unexpected keys: {unexpected}")
    else:
        discarded = len(incompatible.unexpected_keys)
        if discarded:
            print(f"  Discarded {discarded} running-stats buffers (expected, not needed at inference)")
    net.eval()
    print("  Weights loaded OK")

    # --- Quick inference check before export ---
    dummy = torch.zeros(1, 3, 256, 256)
    with torch.no_grad():
        out = net(dummy)
    assert out.shape == (1, 3, 256, 256)
    assert -1.0 <= out.min().item() and out.max().item() <= 1.0
    print(f"  Inference check OK — output range [{out.min():.3f}, {out.max():.3f}]")

    # --- ONNX export ---
    onnx_path = ONNX_DIR / f"{name}.onnx"
    torch.onnx.export(
        net,
        dummy,
        str(onnx_path),
        export_params=True,
        opset_version=17,
        do_constant_folding=True,
        input_names=["input"],
        output_names=["output"],
        dynamic_axes={
            "input":  {2: "height", 3: "width"},
            "output": {2: "height", 3: "width"},
        },
    )

    size_mb = onnx_path.stat().st_size / 1_048_576
    print(f"  Exported → {onnx_path.name} ({size_mb:.1f} MB)")
    return onnx_path


onnx_paths: dict[str, Path] = {}
for name, pth_path in pth_paths.items():
    try:
        onnx_paths[name] = export_to_onnx(name, pth_path)
    except Exception as exc:
        print(f"  *** Export failed for {name}: {exc} ***")

print(f"\nExported {len(onnx_paths)}/{len(pth_paths)} models successfully.")


## Validate ONNX models with ONNX Runtime

Runs each exported model through ONNX Runtime on a 256×256 test image
and checks that the output is in the expected `[-1, 1]` range.

In [ ]:
# Install onnxruntime if not already available (Kaggle does not pre-install it)
try:
    import onnxruntime  # noqa: F401
    print(f"onnxruntime {onnxruntime.__version__} already installed")
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "onnxruntime"])
    print("onnxruntime installed")


In [ ]:
import onnxruntime as ort

print(f"ONNX Runtime {ort.__version__}")


def validate_onnx(name: str, onnx_path: Path) -> None:
    """Run a forward pass through the ONNX model and check output range."""
    sess = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
    input_name = sess.get_inputs()[0].name

    # Random RGB image in [-1, 1]
    test_input = np.random.uniform(-1, 1, (1, 3, 256, 256)).astype(np.float32)
    output = sess.run(None, {input_name: test_input})[0]

    assert output.shape == (1, 3, 256, 256), f"Unexpected shape: {output.shape}"
    o_min, o_max = float(output.min()), float(output.max())
    assert -1.0 <= o_min and o_max <= 1.0, f"Output out of range: [{o_min:.3f}, {o_max:.3f}]"
    print(f"  {name}: PASS — shape {output.shape}, range [{o_min:.3f}, {o_max:.3f}]")


print("Validating with ONNX Runtime:")
for name, onnx_path in onnx_paths.items():
    try:
        validate_onnx(name, onnx_path)
    except Exception as exc:
        print(f"  {name}: FAIL — {exc}")

print("\nValidation complete.")

## Visual spot-check (optional)

Apply the Monet model to a small test image and display the result.

In [ ]:
import matplotlib.pyplot as plt

SPOT_CHECK_MODEL = "style_monet"

if SPOT_CHECK_MODEL not in onnx_paths:
    print(f"{SPOT_CHECK_MODEL} not available — skipping visual check.")
else:
    # Create a synthetic test image (gradient + noise) so we need no input file
    h, w = 256, 256
    grad_x = np.linspace(0, 1, w, dtype=np.float32)[np.newaxis, :] * np.ones((h, 1), dtype=np.float32)
    grad_y = np.linspace(0, 1, h, dtype=np.float32)[:, np.newaxis] * np.ones((1, w), dtype=np.float32)
    test_img_np = np.stack([
        grad_x,
        grad_y,
        (grad_x + grad_y) / 2,
    ], axis=2)  # H×W×3, range [0, 1]

    # Normalise to [-1, 1], add batch dim → [1, 3, H, W]
    inp = (test_img_np.transpose(2, 0, 1)[np.newaxis, ...] * 2.0 - 1.0).astype(np.float32)

    sess = ort.InferenceSession(
        str(onnx_paths[SPOT_CHECK_MODEL]),
        providers=["CPUExecutionProvider"],
    )
    out = sess.run(None, {sess.get_inputs()[0].name: inp})[0][0]  # [3, H, W]

    # De-normalise to [0, 1] for display
    out_img = np.clip((out.transpose(1, 2, 0) + 1.0) / 2.0, 0, 1)

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].imshow(test_img_np)
    axes[0].set_title("Input (synthetic gradient)")
    axes[0].axis("off")
    axes[1].imshow(out_img)
    axes[1].set_title(f"Output — {SPOT_CHECK_MODEL}")
    axes[1].axis("off")
    plt.tight_layout()
    plt.savefig("/kaggle/working/spot_check.png", dpi=100)
    plt.show()
    print("Spot check image saved to /kaggle/working/spot_check.png")

## Output files summary

Lists all generated ONNX files with their sizes.
Download these four files from the Kaggle output panel (right side).

**Installation:** For each file, run `add_GAN_style.ipynb` locally and select
layout **`CycleGAN (PyTorch NCHW tanh)`** (`nchw_tanh`).

In [ ]:
print("=" * 55)
print("Generated ONNX files ready for download:")
print("=" * 55)

total = 0
for name in MODELS:
    p = ONNX_DIR / f"{name}.onnx"
    if p.exists():
        size_mb = p.stat().st_size / 1_048_576
        total += p.stat().st_size
        print(f"  {p.name:<25}  {size_mb:6.1f} MB")
    else:
        print(f"  {name}.onnx{'':20}  *** MISSING ***")

print("-" * 55)
print(f"  {'Total':<25}  {total / 1_048_576:6.1f} MB")
print()
print("tensor_layout for each: nchw_tanh")
print("Install via: scripts/add_GAN_style.ipynb")